# **Cross-dataset deduplication (Step 1) — SHA-256 + pHash check between the tooth-segmentation dataset (train+valid) and the full caries pool. Generates imagenes_excluidas_por_fuga.csv, consumed by Tratado_2.**

## CELL 1 — Install the drive, etc.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imagehash pillow


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.3 MB/s eta 0:00:00


## CELL 2 — Routes

In [ ]:
from pathlib import Path

BASE_DIENTES = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET DIENTES")
BASE_CARIES  = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES")

RUTAS_SEGMENTADOR = [
    BASE_DIENTES / "train" / "images",
    BASE_DIENTES / "valid" / "images",
]

RUTAS_CARIES = [
    BASE_CARIES / "dataset_caries_corregido" / "images" / "train",
    BASE_CARIES / "dataset_caries_corregido" / "images" / "val",
    BASE_CARIES / "dataset_caries_corregido" / "images" / "test",
]


## CELL 3 — Listen to both sides

In [ ]:
import hashlib, imagehash
from PIL import Image
import pandas as pd

def sha256_de(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()

def phash_de(ruta):
    try:
        return str(imagehash.phash(Image.open(ruta)))
    except Exception:
        return None

def tabla_hashes(carpetas, etiqueta):
    filas = []
    for carpeta in carpetas:
        for r in sorted(carpeta.glob("*")):
            if r.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            filas.append({
                "archivo": r.name, "ruta": str(r), "origen": etiqueta,
                "sha256": sha256_de(r), "phash": phash_de(r),
            })
    return pd.DataFrame(filas)

df_seg = tabla_hashes(RUTAS_SEGMENTADOR, "segmentador_train_val")
df_car = tabla_hashes(RUTAS_CARIES, "caries_pool_completo")
print(f"Segmentador (train+val): {len(df_seg)} imágenes")
print(f"Caries (pool completo): {len(df_car)} imágenes")


Segmentador (train+val): 1760 imágenes
Caries (pool completo): 1262 imágenes


## CELL 4 — Exact matches (SHA-256)

In [ ]:
coincid_exactas = df_seg.merge(df_car, on="sha256", suffixes=("_seg", "_caries"))
print(f"Coincidencias exactas: {len(coincid_exactas)}")
coincid_exactas[["archivo_seg", "archivo_caries", "sha256"]]


Coincidencias exactas: 187


,archivo_seg,archivo_caries,sha256
0,c_Frontal_anonymous-frontalView-1727270503644_...,pilot_Frontal_anonymous-frontalView-1727270503...,a3fee9e61a06ed0da488b49af5df270ea5e9bde2d99aa7...
1,c_Frontal_anonymous-frontalView-1727270503644_...,pilot_Frontal_anonymous-frontalView-1727270503...,a3fee9e61a06ed0da488b49af5df270ea5e9bde2d99aa7...
2,c_Frontal_anonymous-frontalView-1727759245618_...,pilot_Frontal_anonymous-frontalView-1727759245...,fabf8de0f8327d8200707bc8bbe751cdeda51c2d91608e...
3,c_Frontal_anonymous_003-007-1141-01_1732595713...,retractors_Frontal_anonymous_003-007-1141-01_1...,127acecc9ea38b70393ad8988f7f03780d83a95c6be0a8...
4,c_Frontal_anonymous_003-007-608-01_17290615569...,retractors_Frontal_anonymous_003-007-608-01_17...,dfbbf95e83645f92ed2a8fbd48cdfedffe8edd7495ce5f...
...,...,...,...
182,c_Maxillary_Occlusal_anonymous_003-008-1371-01...,retractors_Maxillary_Occlusal_anonymous_003-00...,b0776238dfe9f3848915feabcf84a567456945eae93455...
183,c_Maxillary_Occlusal_anonymous_003-008-626-01_...,retractors_Maxillary_Occlusal_anonymous_003-00...,5e1ad86f73268d2fcdaf4c07f79e71b95b68876dc8e5b7...
184,c_Maxillary_Occlusal_anonymous_003-008-649-01_...,retractors_Maxillary_Occlusal_anonymous_003-00...,2429e9db78376a96e28af4a0f49c95e3beeebbbc3b50cf...
185,c_Maxillary_Occlusal_anonymous_003_007_361_01_...,retractors_Maxillary_Occlusal_anonymous_003_00...,a41164b838eff1497983a642d242c4868366b82ee3f34d...


## CELL 5 — Perceptual matches (pHash, Hamming distance ≤ 5)

In [ ]:
def cruzar_phash(a, b, umbral=5):
    filas = []
    hashes_b = [(idx, imagehash.hex_to_hash(h)) for idx, h in b["phash"].items() if h]
    for idx_a, ha_str in a["phash"].items():
        if not ha_str:
            continue
        ha = imagehash.hex_to_hash(ha_str)
        for idx_b, hb in hashes_b:
            dist = ha - hb
            if dist <= umbral:
                filas.append({
                    "archivo_seg": a.loc[idx_a, "archivo"],
                    "archivo_caries": b.loc[idx_b, "archivo"],
                    "distancia": dist,
                })
    return pd.DataFrame(filas)

coincid_phash = cruzar_phash(df_seg, df_car)
print(f"Coincidencias perceptuales (pHash <= 5): {len(coincid_phash)}")
coincid_phash.sort_values("distancia")


Coincidencias perceptuales (pHash <= 5): 202


,archivo_seg,archivo_caries,distancia
0,c_Frontal_anonymous-frontalView-1727270503644_...,pilot_Frontal_anonymous-frontalView-1727270503...,0
1,c_Frontal_anonymous-frontalView-1727270503644_...,pilot_Frontal_anonymous-frontalView-1727270503...,0
2,c_Frontal_anonymous-frontalView-1727759245618_...,pilot_Frontal_anonymous-frontalView-1727759245...,0
3,c_Frontal_anonymous_003-007-1141-01_1732595713...,retractors_Frontal_anonymous_003-007-1141-01_1...,0
4,c_Frontal_anonymous_003-007-608-01_17290615569...,retractors_Frontal_anonymous_003-007-608-01_17...,0
...,...,...,...
40,c_Mandibular_anonymous_003-007-631-01_17291433...,retractors_Mandibular_anonymous_003-007-631-01...,2
61,c_Mandibular_anonymous_003-008-1113-01_1732361...,retractors_Mandibular_anonymous_003-008-1113-0...,2
82,c_Mandibular_anonymous_003-008-730-01_17295200...,retractors_Mandibular_anonymous_003-008-730-01...,4
119,c_Maxillary_Occlusal_anonymous-maxillaryView-1...,pilot_Maxillary_Occlusal_anonymous-maxillaryVi...,4


## CELL 6 — Consolidate and export the exclusion list

In [ ]:
excluir = set(coincid_exactas["archivo_caries"]) | set(coincid_phash["archivo_caries"])
print(f"Total de imágenes del lado de CARIES a excluir: {len(excluir)}")

pd.Series(sorted(excluir), name="archivo_a_excluir").to_csv(
    BASE_CARIES / "imagenes_excluidas_por_fuga.csv", index=False
)
print(f"Guardado en: {BASE_CARIES / 'imagenes_excluidas_por_fuga.csv'}")


Total de imágenes del lado de CARIES a excluir: 180
Guardado en: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/imagenes_excluidas_por_fuga.csv
